# Build postfire atlas on CyVerse

Run this in a **VICE Jupyter Lab** session so the 6 GB MESMA fractions file is read from the data store. Only a few MB of atlas files are written out.

Default input:
`/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/mesma/11_S_LA_2026_7_28/11_S_LA_2026_7_28_mesma_20260903T080145_fractions`

Upload `build_atlas_cyverse.py` and `postfire_landscape_atlas1.html` into the same folder as this notebook, then run the cell below.

In [6]:
import json, shutil, sys
from pathlib import Path

here = Path.cwd()
sys.path.insert(0, str(here))
import build_atlas_cyverse as bac

root = None
for cand in [
    Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28'),
    Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28'),
    Path('/home/jovyan/data-store/new_data/2026-07-28'),
]:
    if cand.exists():
        root = cand
        break
if root is None:
    raise FileNotFoundError('Could not find new_data/2026-07-28')

atlas = root / 'atlas'
mesma = root / 'mesma'
tsf = root.parent / 'Time_since_fire'
html_src = here / 'postfire_landscape_atlas1.html'
scenes = [p for p in atlas.iterdir() if p.is_dir() and (p / 'mesma_grid.js').exists()] if atlas.exists() else []
print('atlas scenes', len(scenes))

if html_src.exists():
    text = html_src.read_text()
    for d in scenes:
        (d / 'postfire_landscape_atlas1.html').write_text(text)
    print('copied HTML into', len(scenes), 'tiles')

def load_feats(path):
    try:
        if not path.exists() or path.stat().st_size < 100:
            return []
        feats = json.loads(path.read_text()).get('features') or []
        print(f'  {path.name}: {len(feats)}')
        return feats
    except OSError as exc:
        print('  skip', path.name, exc)
        return []

def feat_key(feat):
    props = feat.get('properties') or {}
    name = (props.get('Incid_Name') or '').strip()
    year = props.get('burn_year')
    date = props.get('burn_date')
    if name:
        return ('named', name, year, date)
    geom = feat.get('geometry')
    return ('poly', year, json.dumps(geom, separators=(',', ':')) if geom else None)

sources = []
for folder, names in [
    (tsf / 'by_year_2015_2020', ('fires_2015_2020.geojson', 'fires.geojson')),
    (tsf / 'by_year_2015_2026', ('fires_2021_2026.geojson', 'fires.geojson')),
]:
    hit = next((folder / n for n in names if (folder / n).exists() and (folder / n).stat().st_size > 1000), None)
    if hit:
        sources.append(hit)
    else:
        sources.extend(sorted(folder.glob('*_mtbs_perimeters.geojson')))
        sources.extend(sorted(folder.glob('*_modis_perimeters.geojson')))

slim, seen = [], set()
print('merging fire files:')
for src in sources:
    for feat in load_feats(src):
        if not feat.get('geometry'):
            continue
        key = feat_key(feat)
        if key in seen:
            continue
        seen.add(key)
        slim.append(feat)

dest = atlas / 'tsf_years' / 'fires.geojson'
dest.parent.mkdir(parents=True, exist_ok=True)
payload = json.dumps({'type': 'FeatureCollection', 'features': slim}, separators=(',', ':'))
Path('/tmp/fires_atlas.geojson').write_text(payload)
shutil.copyfile('/tmp/fires_atlas.geojson', dest)
shutil.copyfile('/tmp/fires_atlas.geojson', atlas / 'tsf_years' / 'fires_web.geojson')
years = sorted({(f.get('properties') or {}).get('burn_year') for f in slim} - {None})
print('atlas fires.geojson', len(slim), 'years', years)

bac.copy_tsf_html(atlas)
bac.write_atlas_index(atlas, [p.name for p in scenes], mesma_dir=mesma)
print('Open the same /proxy/8765/index.html link and hard-refresh (Cmd+Shift+R)')
'print('On a scene: Fires 2015–2026 (year dropdown) and fire_grids')

SyntaxError: invalid decimal literal (4233655113.py, line 90)

# Final Link

In [19]:
import json, os, re, socket, subprocess, sys, time, shutil
from pathlib import Path
from IPython.display import display, HTML

here = Path.cwd()
sys.path.insert(0, str(here))
import build_atlas_cyverse as bac
import importlib
importlib.reload(bac)

new_data = None
for cand in [
    Path('/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data'),
    Path('/home/jovyan/data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data'),
    Path('/home/jovyan/data-store/new_data'),
]:
    if cand.exists() and (cand / '2026-07-28').exists():
        new_data = cand
        break
if new_data is None:
    raise FileNotFoundError('Could not find new_data')

print('new_data', new_data)
for p in sorted(new_data.iterdir()):
    if p.is_dir() and re.match(r'\d{4}-\d{2}-\d{2}$', p.name):
        n_mesma = len(list((p / 'mesma').glob('*/*_fractions'))) if (p / 'mesma').exists() else 0
        n_atlas = len(list((p / 'atlas').glob('*/mesma_grid.js'))) if (p / 'atlas').exists() else 0
        print(f'  {p.name}:  {n_mesma} fractions  {n_atlas} atlas scenes')

dates = sorted(
    p.name for p in new_data.iterdir()
    if p.is_dir() and (p / 'atlas').exists() and re.match(r'\d{4}-\d{2}-\d{2}$', p.name)
)
print('atlas dates', dates)
if '2026-10-16' not in dates:
    print('\nOctober atlas not built yet.')
    print('1. Put cubes in', new_data / '2026-10-16' / 'mesma')
    print("2. In cyverse_pipeline_2026_07_28_all.ipynb set DATE = '2026-10-16'")
    print('3. Run inventory + atlas build cells, then re-run this cell.')

fires_src = None
for cand in [
    new_data / 'Time_since_fire' / 'by_year_2015_2026' / 'fires.geojson',
    new_data / 'Time_since_fire' / 'by_year_2015_2020' / 'fires.geojson',
]:
    if cand.exists() and cand.stat().st_size > 1000:
        fires_src = cand
        break
for date in dates:
    dest = new_data / date / 'atlas' / 'tsf_years' / 'fires.geojson'
    dest.parent.mkdir(parents=True, exist_ok=True)
    if fires_src:
        shutil.copyfile(fires_src, dest)
        print('fires →', dest)

if not hasattr(bac, 'copy_atlas_html'):
    raise RuntimeError('Upload the new build_atlas_cyverse.py next to this notebook, then re-run.')

html_src = here / 'postfire_landscape_atlas1.html'
if not html_src.exists():
    html_src = Path(bac.__file__).with_name('postfire_landscape_atlas1.html')
print('HTML source', html_src, 'exists' if html_src.exists() else 'MISSING')
n_html = bac.copy_atlas_html(new_data, html_src, dates)
print('scene HTML pages updated:', n_html)

index = bac.write_multi_date_index(new_data, dates or None)
print('picker', index)

PORT = 8766

def port_open(port):
    with socket.socket() as s:
        s.settimeout(0.3)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(PORT):
    subprocess.Popen(
        ['python3', '-m', 'http.server', str(PORT), '--bind', '127.0.0.1'],
        cwd=str(new_data), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    time.sleep(0.8)

prefix = os.environ.get('JUPYTERHUB_SERVICE_PREFIX', '/')
if not prefix.endswith('/'):
    prefix += '/'
url = f'{prefix}proxy/{PORT}/index.html?v={int(time.time())}'
print(url)
display(HTML(
    f'<p style="font-size:18px"><a href="{url}" target="_blank"><b>Open atlas (July / October dropdown)</b></a></p>'
    '<p>The old cell only rewrote the picker. This copy puts the new HTML into every tile. Hard-refresh (Cmd+Shift+R). Port 8766 from new_data/.</p>'
))

new_data /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data
  2026-07-28:  42 fractions  41 atlas scenes
  2026-10-16:  5 fractions  0 atlas scenes
atlas dates ['2026-07-28']

October atlas not built yet.
1. Put cubes in /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-10-16/mesma
2. In cyverse_pipeline_2026_07_28_all.ipynb set DATE = '2026-10-16'
3. Run inventory + atlas build cells, then re-run this cell.
fires → /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/2026-07-28/atlas/tsf_years/fires.geojson
HTML source /home/jovyan/data-store/sce_site/postfire_landscape_atlas1.html exists
Copied postfire_landscape_atlas1.html (107 KB) into 41 scene page(s)
scene HTML pages updated: 41
Copied postfire_landscape_atlas1.html (107 KB) into 41 scene page(s)
Wrote /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_data/index.html (40 scenes, dates ['2026-07-28'])
picker /data-store/iplant/home/samiksha23/SCE-CU/sentinel2_data/new_da